In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np
from scipy import stats
from procrustes import rotational
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL, AMU2ELECTRON_MASS, HARTREE2WAVENUMBER
from cmm.drivers import OptimizationDriver, HarmonicAnalysisDriver
from cmm.misc_utils import read_xyz, write_xyz, get_masses

import openmm.app as app
from ase.io import read

In [2]:
def rmsd(A: np.ndarray, B: np.ndarray):
    return np.sqrt(np.mean(np.sum((A - B)**2, axis=1)))

In [3]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/ion_water_params_8_19.xml')
ff = ForceFieldXML(ff_path, device=device)

In [4]:
water_cluster_pdb_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')
water_cluster_xyz_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.xyz')

f_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_f_scan.pdb')
cl_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cl_scan.pdb')
br_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_br_scan.pdb')
i_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_i_scan.pdb')

li_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_li_scan.pdb')
na_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_na_scan.pdb')
k_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_k_scan.pdb')
rb_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_rb_scan.pdb')
cs_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cs_scan.pdb')

mg_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_mg_scan.pdb')
ca_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_ca_scan.pdb')

In [5]:
water_cluster_pdb = app.PDBFile(water_cluster_pdb_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

topologies_no_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_no_fd = [ff.parametrize(topology, use_fd_morse=False, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_no_fd]

topologies_with_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_with_fd = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_with_fd]

reference_coords, reference_labels = read_xyz(water_cluster_xyz_path)
reference_masses = [get_masses(reference_labels[i]) for i in range(len(reference_labels))]

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [6]:
def create_systems(pdb_path: str):
    pdb_openmm = app.PDBFile(pdb_path)
    positions = [pdb_openmm.getPositions(True, frame=i)._value * 10.0 for i in range(pdb_openmm.getNumFrames())]
    topologies = [Topology.fromMultiPDB(pdb_path, device, frame_index=i) for i in range(pdb_openmm.getNumFrames())]
    systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]
    return positions, systems

In [7]:
f_water_positions, f_water_scan_systems = create_systems(f_water_scan_pdb_path)
cl_water_positions, cl_water_scan_systems = create_systems(cl_water_scan_pdb_path)
br_water_positions, br_water_scan_systems = create_systems(br_water_scan_pdb_path)
i_water_positions, i_water_scan_systems = create_systems(i_water_scan_pdb_path)

li_water_positions, li_water_scan_systems = create_systems(li_water_scan_pdb_path)
na_water_positions, na_water_scan_systems = create_systems(na_water_scan_pdb_path)
k_water_positions, k_water_scan_systems = create_systems(k_water_scan_pdb_path)
rb_water_positions, rb_water_scan_systems = create_systems(rb_water_scan_pdb_path)
cs_water_positions, cs_water_scan_systems = create_systems(cs_water_scan_pdb_path)

mg_water_positions, mg_water_scan_systems = create_systems(mg_water_scan_pdb_path)
ca_water_positions, ca_water_scan_systems = create_systems(ca_water_scan_pdb_path)

In [8]:
ions_systems_and_positions = [
    ("li+", li_water_positions[7], li_water_scan_systems[7]),
    ("na+", na_water_positions[7], na_water_scan_systems[7]),
    ("k+", k_water_positions[7], k_water_scan_systems[7]),
    ("rb+", rb_water_positions[7], rb_water_scan_systems[7]),
    ("cs+", cs_water_positions[7], cs_water_scan_systems[7]),
    ("mg2+", mg_water_positions[7], mg_water_scan_systems[7]),
    ("ca2+", ca_water_positions[7], ca_water_scan_systems[7]),
    ("f-", f_water_positions[7], f_water_scan_systems[7]),
    ("cl-", cl_water_positions[7], cl_water_scan_systems[7]),
    ("br-", br_water_positions[7], br_water_scan_systems[7]),
    ("i-", i_water_positions[7], i_water_scan_systems[7])
]

def optimize_ion_water_dimers_and_compute_frequencies(data_in):
    optimized_coords = []
    optimized_energies = []
    harmonic_frequencies = []
    for i in range(len(data_in)):
        label, positions, system = data_in[i]
        opt_driver = OptimizationDriver(system)
        coords = torch.from_numpy(positions / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        energy = result.fun * HARTREE2KCAL
        print(f"Optimized H2O...{label}: E = {energy}")
        optimized_coords.append(coords_opt)
        optimized_energies.append(energy)
        
        harmonic_driver = HarmonicAnalysisDriver(system)
        masses = get_masses(system.top._atom_symbols)
        hessian = harmonic_driver.run(coords_opt.to(device).requires_grad_(False), box, masses * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER
        nan_mask = torch.isnan(freqs)
        nan_indices = torch.nonzero(nan_mask)
        freqs[nan_indices] = torch.zeros(nan_indices.size())
        harmonic_frequencies.append(freqs)

    return optimized_coords, optimized_energies, harmonic_frequencies

In [9]:
optimized_ion_water_coords, optimized_ion_water_energies, ion_water_dimer_freqs = optimize_ion_water_dimers_and_compute_frequencies(ions_systems_and_positions)

Optimized H2O...li+: E = -34.80323683085005
Optimized H2O...na+: E = -24.23560967779155
Optimized H2O...k+: E = -17.56764744814811
Optimized H2O...rb+: E = -15.397568517765993
Optimized H2O...cs+: E = -13.86772038289375
Optimized H2O...mg2+: E = -91.74796181876411
Optimized H2O...ca2+: E = -60.78461589204164
Optimized H2O...f-: E = -29.739420695155516
Optimized H2O...cl-: E = -15.375447917979939
Optimized H2O...br-: E = -13.416551028662319
Optimized H2O...i-: E = -11.222972381911028


In [10]:
ion_water_dimer_freqs

[tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 5.0715e-05, 7.7383e+01, 1.4253e+02,
         1.8568e+02, 6.3723e+02, 6.7407e+02, 1.7414e+03, 3.7686e+03, 3.8569e+03]),
 tensor([0.0000e+00, 4.8643e-06, 2.5856e-05, 3.6895e+01, 6.6123e+01, 6.6496e+01,
         2.6208e+02, 3.4751e+02, 5.2702e+02, 1.7195e+03, 3.7823e+03, 3.8752e+03]),
 tensor([0.0000e+00, 0.0000e+00, 2.3583e-05, 2.4672e+01, 4.0765e+01, 5.5480e+01,
         2.3476e+02, 2.8918e+02, 4.5155e+02, 1.7044e+03, 3.7907e+03, 3.8870e+03]),
 tensor([0.0000e+00, 5.6925e-06, 2.0162e-05, 1.9443e+01, 3.1811e+01, 5.2798e+01,
         1.9201e+02, 2.9249e+02, 4.1886e+02, 1.6989e+03, 3.7943e+03, 3.8918e+03]),
 tensor([0.0000e+00, 0.0000e+00, 4.7255e-06, 1.7271e+01, 2.7896e+01, 4.7001e+01,
         1.6677e+02, 2.8920e+02, 4.0559e+02, 1.6953e+03, 3.7970e+03, 3.8953e+03]),
 tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 3.0742e-05, 1.6324e+02, 2.2469e+02,
         2.8900e+02, 8.1842e+02, 1.0074e+03, 1.9085e+03, 3.6073e+03, 3.6643e+03]),
 tensor([0.0000e

In [11]:
i_water_positions[7]

array([[-0.025, -0.402, -0.057],
       [ 0.766,  0.133,  0.029],
       [ 0.294, -1.12 , -0.633],
       [ 2.136, -2.535, -2.171]])

In [12]:
li_rmsd = rmsd(optimized_ion_water_coords[0].cpu().numpy() * BOHR2ANG, li_water_positions[7])
na_rmsd = rmsd(optimized_ion_water_coords[1].cpu().numpy() * BOHR2ANG, na_water_positions[7])
k_rmsd = rmsd(optimized_ion_water_coords[2].cpu().numpy() * BOHR2ANG, k_water_positions[7])
rb_rmsd = rmsd(optimized_ion_water_coords[3].cpu().numpy() * BOHR2ANG, rb_water_positions[7])
cs_rmsd = rmsd(optimized_ion_water_coords[4].cpu().numpy() * BOHR2ANG, cs_water_positions[7])

mg_rmsd = rmsd(optimized_ion_water_coords[5].cpu().numpy() * BOHR2ANG, mg_water_positions[7])
ca_rmsd = rmsd(optimized_ion_water_coords[6].cpu().numpy() * BOHR2ANG, ca_water_positions[7])

f_rmsd = rmsd(optimized_ion_water_coords[7].cpu().numpy() * BOHR2ANG, f_water_positions[7])
cl_rmsd = rmsd(optimized_ion_water_coords[8].cpu().numpy() * BOHR2ANG, cl_water_positions[7])
br_rmsd = rmsd(optimized_ion_water_coords[9].cpu().numpy() * BOHR2ANG, br_water_positions[7])
i_rmsd = rmsd(optimized_ion_water_coords[10].cpu().numpy() * BOHR2ANG, i_water_positions[7])

In [13]:
ion_water_rmsds = np.array([
    li_rmsd, na_rmsd, k_rmsd, rb_rmsd, cs_rmsd, mg_rmsd, ca_rmsd, f_rmsd, cl_rmsd, br_rmsd, i_rmsd
])
print(ion_water_rmsds)

[0.04739965 0.06254817 0.06088999 0.06149645 0.06883859 0.12389464
 0.10562457 0.06970701 0.05360733 0.08315184 0.0599322 ]


In [14]:
coords = torch.from_numpy(positions[1] / BOHR2ANG).to(device).requires_grad_(False)
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[1].getEnergy(coords, box)

for key in energies:
    print(f"{key}: {energies[key] * HARTREE2KCAL} kcal/mol")

bond: 0.35676326831702043 kcal/mol
angle: 0.023446173018395126 kcal/mol
torsion: 0.0 kcal/mol
bond_bond: -0.005611845671081472 kcal/mol
bond_angle: 0.03452899245356382 kcal/mol
angle_angle: 0.0 kcal/mol
torsion_bond: 0.0 kcal/mol
torsion_angle: 0.0 kcal/mol
torsion_angle_angle: 0.0 kcal/mol
perm_elec: -25.909409713076045 kcal/mol
pol: -3.7103559196676787 kcal/mol
ct_direct: -6.5340623624889105 kcal/mol
xpol: -0.6042721511840822 kcal/mol
pauli: 27.69132165343953 kcal/mol
disp: -6.1386630971369645 kcal/mol
total: -14.796315001996248 kcal/mol


In [15]:
opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
coords_opt, labels = read_xyz(opt_coords_path_with_fd)

all_perm_elec = []
all_pauli = []
all_dispersion = []
all_induction = []
all_total = []

for i in range(len(coords_opt)):
    coords = torch.from_numpy(coords_opt[i] / BOHR2ANG).to(device).requires_grad_(False)
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    energies = systems[i].getEnergy(coords, box)

    all_perm_elec.append(energies['perm_elec'].item() * HARTREE2KCAL)
    all_pauli.append(energies['pauli'].item() * HARTREE2KCAL)
    all_dispersion.append(energies['disp'].item() * HARTREE2KCAL)
    all_induction.append((energies['pol'] + energies['ct_direct']).item() * HARTREE2KCAL)
    all_total.append(energies['total'].item() * HARTREE2KCAL)
print(all_perm_elec)
print(all_pauli)
print(all_dispersion)
print(all_induction)
print(all_total)

[-8.607751904867047, -29.296102363079406, -53.13340712320965, -69.63209919679979, -83.76533898419413, -84.6497181683305, -87.19425422091292, -85.48081909454604, -106.97921260863617, -136.7261123046401, -137.73332732188743, -154.33353363528553, -176.60978874514407, -192.82459788749833, -306.15923709163206, -298.85742337146684, -298.57354001010566, -309.2916417198375, -309.2284720429492, -321.30726290103627, -395.1379631976352, -382.3757679444182, -380.22359063313866, -380.52922115132725, -508.20800984208773]
[8.60292772403882, 33.80011569437263, 65.51194823851014, 87.8569141870256, 101.1190180781098, 103.74855552079633, 109.3596927478444, 107.8559952702092, 132.64066964873672, 169.3759556896411, 172.45607436024247, 193.54383452155832, 222.8765145366385, 244.15243153740542, 390.23175954580324, 375.4633960639609, 376.45180566495713, 394.20458526774934, 396.3080187028132, 411.1973604935507, 509.1241220376364, 486.4984318331419, 478.35868425206877, 487.98539901585127, 660.0747386860592]
[-1

In [16]:
for energy in all_total:
    print(energy)

-4.867066621998187
-15.20182230129026
-27.45461145722888
-36.25740971592304
-45.40318501341644
-45.23241672139597
-45.649811104736656
-44.87335570023064
-57.28402208009381
-72.64073163839927
-72.8082257764941
-82.32512408291747
-93.94811610712613
-103.21902791535356
-164.93045850462894
-163.43579812515603
-163.22288055114714
-164.7475794004762
-165.01401685463054
-175.57055494821856
-213.5944001942058
-209.63266652280154
-208.90231547613863
-201.4563527100212
-274.104266471932


In [17]:
def optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords):
    optimized_coords = []
    optimized_energies = []
    rmsds = []
    for i in range(len(systems)):
        opt_driver = OptimizationDriver(systems[i])
        coords = torch.from_numpy(reference_coords[i] / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        optimized_energies.append(result.fun * HARTREE2KCAL)
        if result.success == False:
            print(f"Warning: Optimization of structure {i} did not converge. Final energy was {optimized_energies[-1]} kcal/mol.")
        # Get RMSD and store new coordinates.
        # Align structures first in case they rotated during optimization.
        coords_opt = coords_opt.numpy() * BOHR2ANG
        result = rotational(coords_opt, reference_coords[i])
        optimized_coords.append(result.new_a)
        rmsd_i = rmsd(result.new_a, reference_coords[i])
        rmsds.append(rmsd_i)
        print(f"Structure {i}: Energy = {optimized_energies[-1]} kcal/mol, RMSD = {rmsd_i} Ang.")
    
    return optimized_coords, optimized_energies, rmsds

In [23]:
opt_coords, opt_energies, opt_rmsds = optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords)

# Write out the optimized coords to a file
write_xyz("reference_opt_cmm_with_fd_morse.xyz", reference_labels, opt_coords)

Structure 0: Energy = -4.867140506747001 kcal/mol, RMSD = 0.036496749587889954 Ang.
Structure 1: Energy = -15.206384526724465 kcal/mol, RMSD = 0.06885408076683572 Ang.
Structure 2: Energy = -27.455122932676584 kcal/mol, RMSD = 0.05703519303808381 Ang.
Structure 3: Energy = -36.257658421089275 kcal/mol, RMSD = 0.05671862782914064 Ang.
Structure 4: Energy = -45.40313218144183 kcal/mol, RMSD = 0.04675692473954364 Ang.
Structure 5: Energy = -45.240588280736546 kcal/mol, RMSD = 0.0637896649618314 Ang.
Structure 6: Energy = -45.65036243517163 kcal/mol, RMSD = 0.06451870947241209 Ang.
Structure 7: Energy = -44.87345926296371 kcal/mol, RMSD = 0.06158054695052744 Ang.
Structure 8: Energy = -57.27975871579441 kcal/mol, RMSD = 0.05751719209184388 Ang.
Structure 9: Energy = -72.64075931362785 kcal/mol, RMSD = 0.05575698204667176 Ang.
Structure 10: Energy = -72.8090446496837 kcal/mol, RMSD = 0.056743572690585505 Ang.
Structure 11: Energy = -82.325696439944 kcal/mol, RMSD = 0.05255251390874921 Ang.


In [25]:
for val in opt_rmsds:
    print(val)

0.036496749587889954
0.06885408076683572
0.05703519303808381
0.05671862782914064
0.04675692473954364
0.0637896649618314
0.06451870947241209
0.06158054695052744
0.05751719209184388
0.05575698204667176
0.056743572690585505
0.05255251390874921
0.05185194354893609
0.07677682395968433
0.06366964845289753
0.08324005606917234
0.07347325902894286
0.08237898959205348
0.10050705833855154
0.06122448583251924
0.10215215231504562
0.06946205916800269
0.0822888884417537
0.06829356547297079
0.10299627552225075


In [19]:
def compute_hbond_distance_freq_correlation_for_ref_clusters(systems, optimized_coord_file):
    all_dists = []
    all_freqs = []

    coords_opt, labels = read_xyz(optimized_coord_file)
    masses = [get_masses(labels[i]) for i in range(len(labels))]
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    for geom_index in tqdm(range(len(coords_opt))):
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)

        harmonic_driver = HarmonicAnalysisDriver(systems[geom_index])
        hessian = harmonic_driver.run(torch.from_numpy(coords_opt[geom_index] / BOHR2ANG).to(device).requires_grad_(False), box, masses[geom_index] * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER

        # Find the atom which moves most for each hbond frequency
        minimum_hbond_frequency = 2500.0
        maximum_hbond_frequency = 3800.0
        mask = (freqs >= minimum_hbond_frequency) & (freqs <= maximum_hbond_frequency)
        hbond_indices = torch.nonzero(mask).squeeze()
        hbond_eigvces = eigvecs[:, hbond_indices]
        if hbond_eigvces.dim() == 1:
            hbond_eigvces.unsqueeze_(1)
        hbond_freqs = freqs[hbond_indices]
        if hbond_freqs.ndim == 0:
            hbond_freqs = hbond_freqs[np.newaxis]

        h_indices = torch.zeros(hbond_eigvces.size(-1), dtype=torch.long)
        for i in range(hbond_eigvces.size(-1)):
            mode = hbond_eigvces[:, i].reshape(-1, 3)
            h_index = torch.argmax(torch.linalg.norm(mode, dim=1))
            h_indices[i] = h_index

        # Get the distances between each hydrogen and the other oxygens in the system
        all_O_indices = []
        all_H_indices = []
        for i_geom in range(len(labels)):
            O_indices = []
            H_indices = []
            for i, label in enumerate(labels[i_geom]):
                if label == 'O':
                    O_indices.append(i)
                if label == 'H':
                    H_indices.append(i)
            all_O_indices.append(O_indices)
            all_H_indices.append(H_indices)

        all_O_indices = [torch.tensor(all_O_indices[i]) for i in range(len(all_O_indices))]
        all_H_indices = [torch.tensor(all_H_indices[i]) for i in range(len(all_H_indices))]

        for index in h_indices:
            if reference_labels[geom_index][index] != 'H':
                print("Warning: Found an h-bond frequency where the most mobile atom was not hydrogen.")

        H_atoms_hbond_only = coords_opt[geom_index][h_indices]
        O_atoms = coords_opt[geom_index][all_O_indices[geom_index]]
        if H_atoms_hbond_only.ndim == 1:
            H_atoms_hbond_only = H_atoms_hbond_only[np.newaxis, :]
            
        dist_vecs = H_atoms_hbond_only[:, np.newaxis, :] - O_atoms[np.newaxis, :, :]
        OH_hbond_dists = np.array([np.min(np.linalg.norm(dist_vecs[i], axis=1)) for i in range(dist_vecs.shape[0])])
        #OH_hbond_dists_sorted = np.sort(OH_hbond_dists)[::-1]
        all_dists.append(OH_hbond_dists)#_sorted)
        all_freqs.append(hbond_freqs.numpy())
    return np.concatenate(all_dists), np.concatenate(all_freqs)

In [20]:
#opt_coords_path_no_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_without_fd_morse.xyz')
#all_dists_no_fd, all_freqs_no_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_no_fd, opt_coords_path_no_fd)
#
#opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
#all_dists_with_fd, all_freqs_with_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_with_fd, opt_coords_path_with_fd)

In [21]:
def plot_badger_rule_correlation(r_hbond_no_fd, freqs_hbond_no_fd,
                                 r_hbond_with_fd, freqs_hbond_with_fd,
                                 title="Badger Rule Correlation"):
    r_eq_cmm = 0.9589289
    freq_ave_cmm = (3945.054377 + 3834.691479) / 2

    r_eq_wb97xv = 0.959274
    freq_ave_wb97xv = (3960.83 + 3859.85) / 2

    shifted_r_cmm_no_fd = r_hbond_no_fd - r_eq_cmm
    shifted_freqs_cmm_no_fd = freqs_hbond_no_fd - freq_ave_cmm

    shifted_r_cmm_with_fd = r_hbond_with_fd - r_eq_cmm
    shifted_freqs_cmm_with_fd = freqs_hbond_with_fd - freq_ave_cmm

    slope_no_fd, intercept_no_fd, _, _, _ = stats.linregress(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd)
    slope_with_fd, intercept_with_fd, _, _, _ = stats.linregress(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd)
    
    line_x_no_fd = np.linspace(shifted_r_cmm_no_fd.min(), shifted_r_cmm_no_fd.max(), 100)
    line_y_no_fd = slope_no_fd * line_x_no_fd + intercept_no_fd

    line_x_with_fd = np.linspace(shifted_r_cmm_with_fd.min(), shifted_r_cmm_with_fd.max(), 100)
    line_y_with_fd = slope_with_fd * line_x_with_fd + intercept_with_fd
    
    plt.figure(figsize=(10, 6))
    plt.scatter(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd, alpha=0.7, color='blue', s=50, label="No FD Morse")
    plt.plot(line_x_no_fd, line_y_no_fd, color='blue', linewidth=2, linestyle="--",
             label=f'Linear fit: y = {slope_no_fd:.2f}x + {intercept_no_fd:.2f}')
    
    plt.scatter(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd, alpha=0.7, color='green', s=50, label="With FD Morse")
    plt.plot(line_x_with_fd, line_y_with_fd, color='green', linewidth=2, linestyle="--",
         label=f'Linear fit: y = {slope_with_fd:.2f}x + {intercept_with_fd:.2f}')
    
    plt.xlabel('OH Bond Shift (Å)')
    plt.ylabel('Frequency Shift (cm^-1)')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Linear Regression Results:")
    print(f"Slope No FD: {slope_no_fd:.4f}")
    print(f"Intercept No FD: {intercept_no_fd:.4f}")
    #print(f"R-squared: {r_value**2:.4f}")
    #print(f"Correlation coefficient: {r_value:.4f}")
    #print(f"P-value: {p_value:.4e}")
    #print(f"Standard error: {std_err:.4f}")

In [22]:
#plot_badger_rule_correlation(all_dists_no_fd, all_freqs_no_fd, all_dists_with_fd, all_freqs_with_fd)